<a href="https://colab.research.google.com/github/nikhilreddi02/TransformerNLPClassifier/blob/main/TransformerNLPClasifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import math

In [2]:
sentences = [
    "I love this movie",
    "This movie is amazing",
    "I really enjoyed the film",
    "The movie was excellent",
    "I hated this movie",
    "This movie is terrible",
    "I really disliked the film",
    "The movie was boring"
]

labels = [
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0
]

In [4]:
def tokenize(sentence):
  return sentence.lower().split()

In [6]:
vocab = {
    "<pad>": 0,
    "<unk>": 1
}

for sentence in sentences:
  for word in tokenize(sentence):
    if word not in vocab:
      vocab[word] = len(vocab)

In [7]:
def encode(sentence):
  return [
      vocab.get(word, vocab["<unk>"])
      for word in tokenize(sentence)
  ]

In [8]:
max_length = 5
def pad_sequence(sequence):
  if(len(sequence) < max_length):
    sequence = sequence + [vocab["<pad>"]] * (max_length - len(sequence))
  return sequence[:max_length]

In [10]:
X = torch.tensor([
    pad_sequence(encode(sentence))
    for sentence in sentences
])
y = torch.tensor(labels)

In [11]:
X

tensor([[ 2,  3,  4,  5,  0],
        [ 4,  5,  6,  7,  0],
        [ 2,  8,  9, 10, 11],
        [10,  5, 12, 13,  0],
        [ 2, 14,  4,  5,  0],
        [ 4,  5,  6, 15,  0],
        [ 2,  8, 16, 10, 11],
        [10,  5, 12, 17,  0]])

In [12]:
y

tensor([1, 1, 1, 1, 0, 0, 0, 0])

In [13]:
class PositionalEncoding(nn.Module):
  def __init__(self,d_model,max_length):
    super().__init__()
    position = torch.arange(max_length).unsqueeze(1)
    div_term = torch.exp(torch.arange(0,d_model,2) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_length,d_model)
    pe[:,0::2] = torch.sin(position * div_term)
    pe[:,1::2] = torch.cos(position * div_term)
    self.register_buffer("pe",pe.unsqueeze(0))

In [28]:
class TransformerClassifier(nn.Module):
  def __init__(
      self,
      vocab_size,
      d_model,
      nhead,
      num_layers,
      num_classes,
      max_length
  ):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,d_model)
    self.positional_encoding = PositionalEncoding(d_model,max_length)
    encoder_layer = nn.TransformerEncoderLayer(d_model,nhead,batch_first=True)
    self.transformer = (
        nn.TransformerEncoder(encoder_layer,num_layers)
    )
    self.fc = nn.Linear(d_model,num_classes)
  def forward(self,X):
    X = self.embedding(X)
    X = X + self.positional_encoding.pe[:,:X.size(1)]
    X = self.transformer(X)
    X = X.mean(dim=1)
    X = self.fc(X)
    return X

In [29]:
model = TransformerClassifier(
    vocab_size=len(vocab),
    d_model=128,
    nhead=8,
    num_layers=2,
    num_classes=2,
    max_length=max_length
)

In [30]:
criterion = nn.CrossEntropyLoss()

In [31]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [32]:
epochs = 100
for epoch in range(epochs):
  optimizer.zero_grad()
  outputs = model(X)
  loss = criterion(outputs,y)
  loss.backward()
  optimizer.step()
  if (epoch + 1) % 10 == 0:
    print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")

Epoch: 10, Loss: 0.7041444182395935
Epoch: 20, Loss: 0.3535093069076538
Epoch: 30, Loss: 0.0037148024421185255
Epoch: 40, Loss: 0.0008317987667396665
Epoch: 50, Loss: 0.0005925148725509644
Epoch: 60, Loss: 0.0005094584776088595
Epoch: 70, Loss: 0.00047000552876852453
Epoch: 80, Loss: 0.00040589901618659496
Epoch: 90, Loss: 0.0004049452836625278
Epoch: 100, Loss: 0.0003555973817128688


In [33]:
def predict(sentence):
  encoded = encode(sentence)
  encoded = pad_sequence(encoded)
  x = torch.tensor([encoded])
  model.eval()
  with torch.no_grad():
    output = model(x)
    prediction = torch.argmax(
        output,dim=1
    ).item()
    if prediction ==1:
      return "positive"
    return "Negative"

In [34]:
print(
    "I love this movie:",
    predict("I love this movie")
)

I love this movie: positive


In [35]:
print(
    "This movie is terrible:",
    predict("This movie is terrible")
)


This movie is terrible: Negative


In [36]:
print(
    "I really enjoyed the film:",
    predict("I really enjoyed the film")
)

I really enjoyed the film: positive


In [39]:
print(
    "The movie was boring:",
    predict("The movie was boring")
)

The movie was boring: Negative
